In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning,
                        message='invalid value encountered in divide')


In [2]:
df = pd.read_csv('features/features_one_week.csv', sep=';')
df = df[df['ОстатокНачалоНедели'] % 1 == 0]
df = df[df['Количество'] >= 0]
df = df[df['Количество'] <= 350]
df['НеделяНачало'] = pd.to_datetime(df['НеделяНачало'])
df.shape

(2471151, 20)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
df_unique = df.drop_duplicates(subset='Номенклатура').copy()
tfidf = TfidfVectorizer()
tfidf.fit(df_unique['Номенклатура'])
text_new = []
for i in range(len(df_unique)):
    s = df_unique['Номенклатура'].iloc[i]
    df_1 = pd.DataFrame(tfidf.transform([s]).T.todense())
    df_1 = df_1[df_1.values > 0]
    text_new.append(df_1.mean().iloc[0])
df_unique['tfidf_mean'] = pd.Series(text_new, index=df_unique.index)
df = df.merge(df_unique[['Номенклатура', 'tfidf_mean']], on='Номенклатура', how='left')
del df_unique

In [ ]:
nom_sup = pd.read_csv('features/nomenclature_with_supplier.csv', sep=',')
nom_sup.loc[(nom_sup['supplier'] == 'НК') | nom_sup['supplier'].isna(), 'supplier'] = 'Нет значения'
df = df.merge(nom_sup, how='left', left_on='Номенклатура', right_on='product').drop(columns=['product'])
df['supplier'] = df['supplier'].fillna('Нет значения')
del nom_sup

s = pd.to_datetime(df['ДатаПоследнегоПоступления'], errors='coerce')
df['МесяцПоследнегоПоступления'] = s.dt.month
df['ГодПоследнегоПоступления']   = s.dt.year
df.drop(columns=['ДатаПоследнегоПоступления'], inplace=True)

In [5]:
# Агрегация недель 
df['КварталНачало'] = df['НеделяНачало'].dt.to_period('Q').dt.start_time

df_q = df.groupby(['КодТовара', 'КварталНачало']).agg({
    'Номенклатура':               'last',
    'Папка1':                     'last',
    'Папка2':                     'last',
    'Поставщик':                  'last',
    'ЕдиницаИзмерения':           'last',
    'ТоварнаяКатегория':          'last',
    'Количество':                 'sum',
    'Розничная30%':               'last',
    'ЗакупочнаяЦена':             'last',
    'ОстатокНачалоНедели':        'first',
    'temp_mean_week':             'mean',
    'precip_sum_week':            'sum',
    'temp_max_week':              'max',
    'temp_min_week':              'min',
    'НДС':                        'last',
    'days_off_in_week':           'sum',
    'tfidf_mean':                 'last',
    'supplier':                   'last',
    'МесяцПоследнегоПоступления': 'last',
    'ГодПоследнегоПоступления':   'last',
}).reset_index().rename(columns={
    'ОстатокНачалоНедели': 'ОстатокНачалоКвартала',
    'temp_mean_week':   'temp_mean_q',
    'precip_sum_week':  'precip_sum_q',
    'temp_max_week':    'temp_max_q',
    'temp_min_week':    'temp_min_q',
    'days_off_in_week': 'days_off_q',
})

print(f'Строк: {len(df_q)}, кварталов: {df_q["КварталНачало"].nunique()}')
del df; gc.collect()

Строк: 249146, кварталов: 35


47

In [6]:
# Матрица квартальных продаж
df_q['КодТовара']     = df_q['КодТовара'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df_q['КварталНачало'] = pd.to_datetime(df_q['КварталНачало']).dt.tz_localize(None).dt.normalize()

df_sells = (
    df_q.set_index(['КодТовара', 'КварталНачало'])[['Количество']]
        .unstack(level=-1).fillna(0)
        .sort_index().sort_index(axis=1)
)
df_sells.columns = df_sells.columns.get_level_values(1)
df_sells.columns = pd.to_datetime(df_sells.columns, errors='coerce').tz_localize(None).normalize()
df_sells.index = df_sells.index.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df_sells.shape

(35442, 35)

In [7]:
def get_timespan_q(df, dt, minus, periods):
    """minus и periods — в кварталах"""
    cols = pd.date_range(
        pd.to_datetime(dt) - pd.DateOffset(months=3 * minus),
        periods=periods, freq='QS'
    )
    return df.reindex(columns=cols, fill_value=0)


def prepare_quarter(df, t):
    X = {}
    # Лаговые фичи по окнам [1, 2, 4, 8] кварталов
    for i in [1, 2, 4, 8]:
        tmp = get_timespan_q(df, t, i, i)
        X[f'diff_{i}q_mean']  = tmp.diff(axis=1).mean(axis=1).values
        X[f'mean_{i}q_decay'] = (tmp * np.power(0.9, np.arange(i)[::-1])).sum(axis=1).values
        X[f'mean_{i}q']       = tmp.mean(axis=1).values
        X[f'median_{i}q']     = tmp.median(axis=1).values
        X[f'min_{i}q']        = tmp.min(axis=1).values
        X[f'max_{i}q']        = tmp.max(axis=1).values
        X[f'std_{i}q']        = tmp.std(axis=1).values

    for i in [1, 2, 4, 8]:
        tmp = get_timespan_q(df, pd.to_datetime(t) - pd.DateOffset(months=3), i, i)
        X[f'diff_{i}q_mean_2']  = tmp.diff(axis=1).mean(axis=1).values
        X[f'mean_{i}q_decay_2'] = (tmp * np.power(0.9, np.arange(i)[::-1])).sum(axis=1).values
        X[f'mean_{i}q_2']       = tmp.mean(axis=1).values
        X[f'median_{i}q_2']     = tmp.median(axis=1).values
        X[f'min_{i}q_2']        = tmp.min(axis=1).values
        X[f'max_{i}q_2']        = tmp.max(axis=1).values
        X[f'std_{i}q_2']        = tmp.std(axis=1).values

    s1 = get_timespan_q(df, t, 4, 1).values.flatten()
    s2 = get_timespan_q(df, t, 8, 1).values.flatten()
    X['sales_1y_ago'] = s1
    X['sales_2y_ago'] = s2
    X['yoy_ratio']    = np.where(s2 > 0, s1 / (s2 + 1e-6), 0.0)
    return pd.DataFrame(X)

In [8]:
# Лаговые фичи
quarters = df_q['КварталНачало'].dropna().drop_duplicates().sort_values().tolist()
print(f'Всего кварталов: {len(quarters)}')

out_frames = []
for q in quarters:
    feats = prepare_quarter(df_sells, q)
    if isinstance(feats.index, pd.RangeIndex):
        feats.index = df_sells.index
    feats.index.name = 'КодТовара'
    feats = feats.reset_index()
    feats['КодТовара']     = feats['КодТовара'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    feats['КварталНачало'] = q

    df_m = df_q[df_q['КварталНачало'] == q].copy()
    if df_m.empty:
        continue
    df_m = df_m.merge(feats, on=['КодТовара', 'КварталНачало'], how='left')
    out_frames.append(df_m)
    del feats, df_m

df = pd.concat(out_frames, ignore_index=True)
del out_frames, df_q, df_sells; gc.collect()
print(f'Итого строк: {len(df)}')

Всего кварталов: 35
Итого строк: 249146


In [9]:
# Фильтрация
df = df[(df['Папка1'] != 'разобрать')]
df = df[df['ЕдиницаИзмерения'] == 'шт'].drop('ЕдиницаИзмерения', axis=1)
df = df[df['ТоварнаяКатегория'] == 'Штучный товар'].drop('ТоварнаяКатегория', axis=1)
# Проверяем оба столбца на дробные значения
bad_cols = ['ОстатокНачалоКвартала', 'Количество']
bad_codes = df.loc[(df[bad_cols].astype(float) % 1 != 0).any(axis=1), 'КодТовара'].unique()
df = df[~df['КодТовара'].isin(bad_codes)]
df = df[df['Номенклатура'] != '1']

df = df[df['КварталНачало'] > '2018-01-01']
df = df[df['Количество'] >= 0]
df = df[df['Количество'] <= 175]
df = df.sort_values('КварталНачало', ascending=True, kind='mergesort')
print(f'После фильтрации: {len(df)}')


После фильтрации: 235067


In [10]:
# Train/Test split
X = df.drop('Количество', axis=1).copy()
y = df['Количество'].copy()

X['КварталНачало'] = pd.to_datetime(X['КварталНачало'])
X_test  = X[X.КварталНачало >= '2025-07-01'].copy()
X_train = X[X.КварталНачало <  '2025-07-01'].copy()
y_test  = y[y.index.isin(X_test.index)]
y_train = y[y.index.isin(X_train.index)]

for frame in [X_test, X_train, X]:
    frame['ТекущийМесяц']   = pd.to_datetime(frame['КварталНачало'], errors='coerce').dt.month
    frame['ТекущийГод']     = pd.to_datetime(frame['КварталНачало'], errors='coerce').dt.year
    frame['ТекущийКвартал'] = pd.to_datetime(frame['КварталНачало'], errors='coerce').dt.quarter
    frame.drop(columns=['КварталНачало'], inplace=True)

object_cols = ['Папка1', 'Папка2', 'Поставщик', 'supplier',
               'МесяцПоследнегоПоступления', 'ГодПоследнегоПоступления',
               'ТекущийМесяц', 'ТекущийГод', 'ТекущийКвартал']
for frame in [X, X_test, X_train]:
    frame[object_cols] = frame[object_cols].astype(object)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 208951, Test: 26116


In [11]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from category_encoders.one_hot import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from xgboost import XGBRegressor


# WAPE
def wape(y_true, y_pred):
    """WAPE = sum(|y_true - y_pred|) / sum(|y_true|)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom  = np.sum(np.abs(y_true))
    if denom == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / denom


wape_scorer = make_scorer(wape, greater_is_better=False)


cols_for_ohe = [x for x in object_cols if X_train[x].nunique() < 5]
cols_for_mte = [x for x in object_cols if X_train[x].nunique() >= 5]
numeric_cols  = list(X_train.select_dtypes(exclude='object').columns)

cols_for_ohe_idx = [list(X_train.columns).index(c) for c in cols_for_ohe]
cols_for_mte_idx = [list(X_train.columns).index(c) for c in cols_for_mte]
numeric_cols_idx = [list(X_train.columns).index(c) for c in numeric_cols]

col_transform = ColumnTransformer([
    ('OHE', OneHotEncoder(),  cols_for_ohe_idx),
    ('MTE', TargetEncoder(),  cols_for_mte_idx),
    ('SC',  StandardScaler(), numeric_cols_idx),
])
col_transform.fit(X_train, y_train)

pipe = Pipeline([
    ('column_transformer', col_transform),
    ('gradient_boosting', XGBRegressor(
        objective='reg:absoluteerror',
        random_state=42,
        n_estimators=300,
        subsample=1.0,
        min_child_weight=1,
        max_depth=6,
        learning_rate=0.1,
        gamma=1
    ))
])
pipe.fit(X_train, y_train)

train_preds = pipe.predict(X_train)
test_preds  = pipe.predict(X_test)
print('Без CV (гиперпараметры подобраны из головы)')
print(f'WAPE train : {wape(y_train, train_preds):.4f}, WAPE test: {wape(y_test, test_preds):.4f}')


Без CV (гиперпараметры подобраны из головы)
WAPE train : 0.4733, WAPE test: 1.4678


In [12]:
# GridSearchCV (CV-метрика — WAPE, 4 комбо x 3 фолда = 12 фитов)
from sklearn.model_selection import GridSearchCV

param_grid = {
    'gradient_boosting__n_estimators': [300, 600],
    'gradient_boosting__max_depth':    [4, 6],
}
cv_fast = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(pipe, param_grid, cv=cv_fast,
                      scoring=wape_scorer,
                      n_jobs=1, verbose=3)
search.fit(X_train, y_train)

print(f'Best params (CV WAPE={-search.best_score_:.5f}):')
print(search.best_params_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print(f'WAPE лучшей модели на тесте: {wape(y_test, y_pred):.5f}')


Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV 1/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.609 total time=   5.2s
[CV 2/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.695 total time=   9.9s
[CV 3/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.701 total time=  15.1s
[CV 1/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.617 total time=  10.6s
[CV 2/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.728 total time=  19.2s
[CV 3/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.742 total time=  28.4s
[CV 1/3] END gradient_boosting__max_depth=6, gradient_boosting__n_estimators=300;, score=-0.613 total time=   6.2s
[CV 2/3] END gradient_boosting__max_depth=6, gradient_boosting__n_estimators=300;, score=-0.765 total time=  11.0s
[CV 3/3] END gradien

In [13]:
cv_results = pd.DataFrame(search.cv_results_)

fold_cols = [c for c in cv_results.columns
             if c.startswith('split') and c.endswith('_test_score')]
keep_cols = (['param_gradient_boosting__n_estimators',
              'param_gradient_boosting__max_depth']
             + fold_cols
             + ['mean_test_score', 'std_test_score', 'rank_test_score'])

df_cv = cv_results[keep_cols].copy()

for c in fold_cols + ['mean_test_score']:
    df_cv[c] = -df_cv[c]

df_cv = df_cv.rename(columns={
    'param_gradient_boosting__n_estimators': 'n_estimators',
    'param_gradient_boosting__max_depth':    'max_depth',
    'mean_test_score': 'WAPE_mean',
    'std_test_score':  'WAPE_std',
    'rank_test_score': 'rank',
    **{c: f'fold{i+1}_WAPE' for i, c in enumerate(fold_cols)},
})
df_cv = df_cv.sort_values('rank').reset_index(drop=True).round(5)

print("CV-результаты по фолдам (TimeSeriesSplit, n_splits=3):")
print(df_cv.to_string(index=False))
df_cv


CV-результаты по фолдам (TimeSeriesSplit, n_splits=3):
 n_estimators  max_depth  fold1_WAPE  fold2_WAPE  fold3_WAPE  WAPE_mean  WAPE_std  rank
          300          4     0.60938     0.69497     0.70120    0.66851   0.04189     1
          600          4     0.61749     0.72774     0.74159    0.69561   0.05552     2
          300          6     0.61261     0.76492     0.78516    0.72090   0.07701     3
          600          6     0.61688     0.77924     0.81291    0.73634   0.08558     4


,n_estimators,max_depth,fold1_WAPE,fold2_WAPE,fold3_WAPE,WAPE_mean,WAPE_std,rank
0,300,4,0.60938,0.69497,0.70120,0.66851,0.04189,1
1,600,4,0.61749,0.72774,0.74159,0.69561,0.05552,2
2,300,6,0.61261,0.76492,0.78516,0.72090,0.07701,3
3,600,6,0.61688,0.77924,0.81291,0.73634,0.08558,4


In [14]:
# Финальное качество на train и test
final_model = best_model

train_preds_final = final_model.predict(X_train)
test_preds_final  = final_model.predict(X_test)

wape_train_final = wape(y_train, train_preds_final)
wape_test_final  = wape(y_test,  test_preds_final)

print('=' * 60)
print('ФИНАЛЬНОЕ КАЧЕСТВО XGBoost (3 месяца)')
print('=' * 60)
print(f'  WAPE train: {wape_train_final:.4f}  (n={len(y_train):,})')
print(f'  WAPE test:  {wape_test_final:.4f}  (n={len(y_test):,})')


ФИНАЛЬНОЕ КАЧЕСТВО XGBoost (3 месяца)
  WAPE train: 0.5447  (n=208,951)
  WAPE test:  1.3799  (n=26,116)


In [15]:
# Проверка адекватности модели: R² на train и test
#   R² = 1  — идеальное предсказание;
#   R² = 0  — модель не лучше предсказания среднего;
#   R² < 0  — модель хуже тривиального прогноза.
from sklearn.metrics import r2_score

r2_train = r2_score(y_train, final_model.predict(X_train))
r2_test  = r2_score(y_test,  final_model.predict(X_test))

print(f"R² train: {r2_train:.4f}")
print(f"R² test:  {r2_test:.4f}")


R² train: 0.5734
R² test:  0.3664


In [ ]:
import joblib
save_path = 'models/3month.pkl'
joblib.dump(best_model, save_path)
print(f'Модель сохранена: {save_path}')

Модель сохранена: D:\learning_projects\uir_all\models\3month.pkl
